# Module 5 | Class 4 Assignment: t-SNE and UMAP Visualization

**Note:** This is a continuation of the Class 3 notebook (Activity 2, Part 2). The same `X_scaled` and `cluster_labels` from Class 3 are used here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

## Setup: Load Data and Prepare (from Class 3)

In [ ]:
# Load dataset
df = pd.read_csv('Mall_Customers.csv')

# Select features
feature_names = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[feature_names].values

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Cluster labels from K-Means (Activity 1 / Class 1)
kmeans = KMeans(n_clusters=5, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)

# PCA 2D (from Class 3 — needed for side-by-side comparison in Task 3)
pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_scaled)

print("Data ready. Shape:", X_scaled.shape)

## Task 1: Apply t-SNE

In [ ]:
# Step 1: Run t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_scaled)

# Step 2: Plot colored by cluster labels
plt.figure(figsize=(6, 5))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_labels, cmap='viridis', alpha=0.8)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.title('Task 1: t-SNE 2D Projection Colored by Cluster Labels')
plt.tight_layout()
plt.show()

**Observation (Step 3):** Compared to the PCA plot from Class 3, t-SNE typically produces tighter and more separated cluster groups because it preserves local neighborhood structure rather than global variance. However, the distances between clusters in t-SNE space do not reflect true distances in the original feature space — only local groupings are meaningful.

## Task 2: Apply UMAP (Optional but Recommended)

In [ ]:
!pip install umap-learn

In [ ]:
import umap

# Step 2: Run UMAP
reducer = umap.UMAP(n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_scaled)

# Step 3: Plot
plt.figure(figsize=(6, 5))
scatter = plt.scatter(X_umap[:, 0], X_umap[:, 1], c=cluster_labels, cmap='viridis', alpha=0.8)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('UMAP Dimension 1')
plt.ylabel('UMAP Dimension 2')
plt.title('Task 2: UMAP 2D Projection Colored by Cluster Labels')
plt.tight_layout()
plt.show()

## Task 3: Compare All Three Methods

In [ ]:
# Step 1: Side-by-side comparison of PCA, t-SNE, and UMAP
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PCA
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', alpha=0.8)
axes[0].set_title('PCA')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# t-SNE
axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_labels, cmap='viridis', alpha=0.8)
axes[1].set_title('t-SNE')
axes[1].set_xlabel('Dim 1')
axes[1].set_ylabel('Dim 2')

# UMAP
axes[2].scatter(X_umap[:, 0], X_umap[:, 1], c=cluster_labels, cmap='viridis', alpha=0.8)
axes[2].set_title('UMAP')
axes[2].set_xlabel('Dim 1')
axes[2].set_ylabel('Dim 2')

plt.suptitle('Task 3: PCA vs t-SNE vs UMAP — Customer Segments', fontsize=14)
plt.tight_layout()
plt.show()

**Comparison: PCA vs t-SNE vs UMAP**

**PCA** is a linear method that projects data onto directions of maximum variance. It is fast, fully deterministic, and the axes (PC1, PC2) are interpretable through loadings. However, because it only captures linear relationships, non-linear cluster structures may not separate well in 2D PCA space.

**t-SNE** is a nonlinear method that preserves local neighborhood structure, making it excellent for revealing tight, visually distinct clusters. On the Mall Customers data, clusters tend to appear more separated than in PCA. The main limitations are that t-SNE is slow on large datasets, stochastic (results vary across runs unless `random_state` is fixed), and inter-cluster distances are not meaningful — two clusters appearing far apart in t-SNE space may not actually be far apart in the original feature space.

**UMAP** is also a nonlinear method but generally faster than t-SNE and better at preserving both local and global structure simultaneously. It tends to produce cleaner cluster separation while keeping the relative positions of clusters more faithful to the original space. UMAP is the preferred choice when both speed and structure preservation matter.

**When to use each:**
- **PCA**: preprocessing step before modeling, when interpretability and speed matter, or when relationships are mostly linear.
- **t-SNE**: exploratory visualization of cluster structure, especially for high-dimensional data like images or text embeddings.
- **UMAP**: when you need fast, scalable nonlinear visualization that also preserves global structure better than t-SNE.